# MDTF Example Multicase POD: Notebook Template

This notebook is both a runnable walkthrough of the `example_multicase` POD and a starting point for new multicase POD notebooks. It follows the same main stages as the POD: resolve the requested variable, load each case, apply MDTF preprocessing, calculate a latitude profile, and plot all cases together.

The defaults use the prepared GFDL cases in this directory. To adapt this notebook, start with the configuration cell and update the case-info file, conventions, requested variable, coordinates, and analysis below. The prepared NetCDF files are large and must be available locally.

## Workflow and template inputs

The POD requests CMIP `tas`; the demo cases contain GFDL `t_ref`. MDTF fieldlist metadata resolves that mapping. For each configured case, the notebook opens its source file, applies the framework's unit and coordinate preprocessing, renames the dependent variable to the POD name, then calculates and plots a time-mean zonal anomaly.

## Notebook import setup

This notebook imports MDTF internals directly from the repository's `src` package. Until a lightweight, pip-installable MDTF API is exposed, it must locate the checkout and add its root to `sys.path` so those imports work when Jupyter is started from a subdirectory. This is a temporary development workaround; consumers of the future installed API should not need to manage repository paths.

In [ ]:
import sys
from pathlib import Path

candidate_dirs = (Path.cwd(), *Path.cwd().parents)
repo_root = next(
    (path for path in candidate_dirs if (path / "src" / "translation.py").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Start Jupyter in the MDTF checkout or one of its subdirectories.")

sys.path.insert(0, str(repo_root))
print(f"MDTF repository: {repo_root}")

## 1. Configure this example

These are the first values to change when reusing the notebook. The case-info YAML supplies one source file and coordinate names for each case.

In [ ]:
EXAMPLE_DIR = repo_root / "diagnostics" / "example_multicase"
CASE_INFO_PATH = EXAMPLE_DIR / "case_info_gfdl_demo.yml"
POD_CONVENTION = "CMIP"
DATA_CONVENTION = "GFDL"
REQUESTED_VAR = "tas"
OUTPUT_DIR = EXAMPLE_DIR / "notebook_work"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Case configuration: {CASE_INFO_PATH}")
print(f"{POD_CONVENTION} {REQUESTED_VAR} using {DATA_CONVENTION} source data")

## 2. Resolve the requested variable

Initialize the same fieldlist translator used by the framework. The minimal POD-like settings below describe the requested field and its dimensions; `Fieldlist.translate` resolves its source name for the selected data convention.

In [ ]:
from types import SimpleNamespace

from src.translation import VariableTranslator
from src.varlist_util import Varlist

translator = VariableTranslator(str(repo_root))
translator.read_conventions(str(repo_root))
requested_field = translator.get_convention(POD_CONVENTION).to_CF(REQUESTED_VAR)

pod_settings = {
    "data": {"frequency": "day", "realm": requested_field.get("realm") or "atmos"},
    "dimensions": {
        "lat": {"axis": "Y", "standard_name": "latitude", "units": "degrees_north"},
        "lon": {"axis": "X", "standard_name": "longitude", "units": "degrees_east"},
        "time": {"axis": "T", "standard_name": "time", "units": "days"},
    },
    "varlist": {
        REQUESTED_VAR: {
            "standard_name": requested_field["standard_name"],
            "long_name": requested_field.get("long_name", ""),
            "units": requested_field["units"],
            "modifier": requested_field.get("modifier", ""),
            "dimensions": ["time", "lat", "lon"],
        }
    },
}
pod = SimpleNamespace(
    name="example_multicase_notebook",
    pod_data=pod_settings["data"],
    pod_dims=pod_settings["dimensions"],
    pod_vars=pod_settings["varlist"],
    pod_settings={"convention": POD_CONVENTION},
    _log_name="example_multicase_notebook",
)
pod_var = Varlist.from_struct(pod).vars[0]
pod_var.translation = translator.get_convention(DATA_CONVENTION).translate(
    pod_var, POD_CONVENTION
)
source_var = pod_var.translation.name
print(f"POD variable: {pod_var.name}; source variable: {source_var}")

## 3. Load each case

The YAML case entries provide input paths and coordinate names. The demo YAML may contain paths from a different checkout, so each file is resolved by its basename under this repository's `prepared_gfdl_data` directory. Change this resolution rule when adapting the notebook to another data layout.

In [ ]:
import xarray as xr
import yaml

with CASE_INFO_PATH.open() as stream:
    case_info = yaml.safe_load(stream)

source_datasets = {}
for case_name, case_config in case_info["CASE_LIST"].items():
    file_key = f"{source_var.upper()}_FILE"
    configured_path = Path(case_config[file_key])
    data_path = EXAMPLE_DIR / "prepared_gfdl_data" / configured_path.name
    if not data_path.is_file():
        raise FileNotFoundError(f"Prepared data file not found: {data_path}")

    dataset = xr.open_dataset(data_path, decode_times=True, use_cftime=True)
    source_datasets[case_name] = dataset
    print(f"{case_name}: {data_path.name}; sizes={dict(dataset.sizes)}")

## 4. Apply MDTF preprocessing

Run the framework's unit conversion and coordinate-renaming functions, then rename the dependent source variable to the name requested by the POD. In this tas/t_ref example, units and coordinate names already agree, so the dependent-variable rename is the visible change.

In [ ]:
from src.preprocessor import (
    ConvertUnitsFunction,
    MDTFPreprocessorBase,
    RenameVariablesFunction,
)

coords_renamed = {}
case_views = {}
for case_name, dataset in source_datasets.items():
    converted = ConvertUnitsFunction().execute(pod_var, dataset)
    coords_renamed[case_name] = RenameVariablesFunction().execute(pod_var, converted)
    case_views[case_name] = SimpleNamespace(
        varlist=SimpleNamespace(iter_vars=lambda: [pod_var])
    )

pod_named_datasets = MDTFPreprocessorBase.rename_dataset_vars(
    None, coords_renamed, case_views
)
for case_name, dataset in pod_named_datasets.items():
    if pod_var.name not in dataset.data_vars:
        raise KeyError(f"{case_name}: expected {pod_var.name!r} after preprocessing")
    print(f"{case_name}: {source_var} -> {pod_var.name}")

## 5. Calculate a profile for each case

For each case, calculate the time mean, subtract its global mean to form an anomaly, and average over longitude to get the zonal-mean profile.

In [ ]:
case_profiles = {}
for case_name, dataset in pod_named_datasets.items():
    case_config = case_info["CASE_LIST"][case_name]
    field = dataset[pod_var.name].mean(dim=case_config["time_coord"])
    anomaly = field - field.mean()
    profile = anomaly.mean(dim=case_config["lon_coord"])
    profile.load()
    case_profiles[case_name] = profile

for dataset in source_datasets.values():
    dataset.close()

print(f"Calculated {len(case_profiles)} case profiles")

## 6. Plot and save the multicase result

The notebook writes its demonstration plot under `notebook_work/`. In a framework-run POD, write figures to the framework-provided `WORK_DIR` output path.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

fig, ax = plt.subplots(figsize=(12, 4))
for case_name, profile in case_profiles.items():
    profile.plot(ax=ax, label=case_name)

ax.set_title("Zonal Mean Surface Air Temperature Anomaly")
ax.set_xlabel("Latitude")
ax.set_ylabel(f"{pod_var.name} anomaly ({pod_var.units})")
ax.legend(title="Case")
fig.tight_layout()
plot_path = OUTPUT_DIR / "example_multicase_notebook.png"
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
display(fig)
plt.close(fig)
print(f"Saved plot: {plot_path}")

## Reuse checklist

- Update `CASE_INFO_PATH` and the source-file path resolution for your cases.
- Set `POD_CONVENTION`, `DATA_CONVENTION`, and `REQUESTED_VAR` for the new POD.
- Update POD dimensions and coordinate keys if the variable is not a time-latitude-longitude field.
- Replace the time mean, anomaly, and zonal mean with the analysis appropriate to the diagnostic.
- For a framework-run POD, write figures to the framework-provided `WORK_DIR`.